# Sistema RSA para firmas digitales

En este notebook, mostraremos el proceso para generar las claves pública y privada, así como el proceso para generar una firma digital y su verificación correspondiente usando el sistema RSA. Empezaremos por cargar las funciones que necesitamos:

In [1]:
import sys
import importlib.util

if 'google.colab' in sys.modules:
    runtime = "Google Colab"
    if importlib.util.find_spec("cryptocalc") is None:
        print("   Installing MA2006B from GitHub\n")
        !pip install git+https://github.com/Krul-dev/MA2006B.git
else:
    runtime = "Local environment"

import cryptocalc

print(f"\n========= Notebook execution context =========")
print(f"              Runtime: {runtime}")
print(f"       Python version: {sys.version.split()[0]}")
print(f"   CryptoCalc version: {cryptocalc.__version__}\n")

from cryptocalc import (
    rsa_key_generation,
    rsa_signature_generation,
    is_valid_rsa_signature,
    sha256_of_sentence,
)


========= Notebook execution context =========
              Runtime: Local environment
       Python version: 3.14.3
   CryptoCalc version: 0.1.0



## Protocolo RSA para la generación de claves

Una vez que ya hemos cargado las librerías necesarias, empezamos por generar nuestras claves *pública* y *privada:*

In [2]:
(public_key, private_key) = rsa_key_generation(2048)
d = private_key[0]
e = public_key[0]
n = public_key[1]

print(f"Exponente público (e): {e}\n")
print(f"Exponente privado (d): {d}\n")
print(f"Módulo para las claves (n): {n}\n")

Exponente público (e): 65537

Exponente privado (d): 27388385966051004195808625923135717564992024854975558357582167666768644339366684437309776946878129788232215844712514241488813406631462556577340635272962349377865031469608481438159021133326624366679627879169963162411217783987415756142876459057028284295662109742948070162492594680913103931265307533837508789143632994027658742359347419013540863054362306325267163347308951327150551005106519585284256443901122662401994422215207134900629319598884808700369257660318735375313654219821897198802100030385111549013668530263549190975505483845717467478026627291579733960390500360799060062220080592110554918547063642488463870486977024213961260080978574910751655780998781292186913846132169344081015455175229373473415641585597051112850727676561019880979538812311599874904434144448487100761523342692744370393235253270396901533362599506393682394834916419453179129067551435859179879671664278095635962907806892037760978808511624653292713640661880434773780291572460109

Notemos que para la generación de estas claves utilizamos números primos generados de manera aleatoria de alrededor de 2048 bits de longitud. Se sigue que el módulo generado $n$ tiene alrededor de 4096 bits de longitud. Es muy importante que el valor del exponente privado $d$ no se divulge. En cambio, el exponente público $e$ y el módulo $n$ forman parte de la clave pública y son conocidos por todos los agentes involucrados, incluída Eva.

## Protocolo RSA para la firma de mensajes

Para ilustrar el protocolo para la generación de firmas digitales, supongamos que Alicia desea firmar el mensaje 'Hello World!'. Empezaremos por generar el hash correspondiente:

In [3]:
m = 'Hello World!'
h = sha256_of_sentence(m)

print(f"Mensaje llano (m): {m}\n")
print(f"Hash del mensaje llano (h): {h}\n")

Mensaje llano (m): Hello World!

Hash del mensaje llano (h): 57676413081093003148005107550719583540116985236696423860923466490497932824681



Con esta información, podemos generar ahora la firma correspondiente y agregarla al mensaje llano para generar el *mensaje firmado:*

In [4]:
s = rsa_signature_generation(private_key, h)
signed_message = (m, s)

print(f"El mensaje firmado es: {signed_message}\n")

El mensaje firmado es: ('Hello World!', 400458020946431784929706337187244298747553933701446639237273338985688829238365216897130435388034467653722731453608535551673131050370004561545479980534697033023405223850687758751337855557363247278042642593448494128726074020366525086197178664156815301082698776754465764416451253391264457543531627430110658424837367446193460193174347007826555336309203970722791779783787148686241888873155102840969478163830470011201401723510372927578631224991147740440041192655219449254727875522731968100075561128610883721319839930480195865352751904177097214016056837601256114063247405408133825128936271908517007225719499506174050748666622891281597880778462615103302632138503700820221238804402812829555824726580184024448723102107633159362356367732212291919843910320717034032716488429833305283728730203115217203182219867972992038918382789142232426777019163203181213020116753174308674931810924812473565748290658492900136792237535305820566848053066579973080200874671513114929643056764

## Protocolo RSA para la verificación de firmas

Para verificar la firma, Beto aplica el protocolo RSA utilizando la clave pública de Alicia, el hash del mensaje y la firma correspondiente:

In [5]:
h = sha256_of_sentence(signed_message[0])
s = signed_message[1]

if is_valid_rsa_signature(public_key,h,s):
    verificacion_firma = "La firma es válida"
else:
    verificacion_firma = "La firma es inválida"

print(verificacion_firma)

La firma es válida
